In [1]:
%pip -q install "datasets==3.6.0" torchaudio tqdm
import datasets
print("datasets:", datasets.__version__)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 8.9 MB/s eta 0:00:00
datasets: 3.6.0


In [2]:
from google.colab import drive
drive.mount("/content/drive")

import os
HF_CACHE = "/content/drive/MyDrive/hf_cache"
os.makedirs(HF_CACHE, exist_ok=True)
os.environ["HF_HOME"] = HF_CACHE
os.environ["HF_DATASETS_CACHE"] = os.path.join(HF_CACHE, "datasets")
os.environ["HF_HUB_CACHE"] = os.path.join(HF_CACHE, "hub")

print("HF cache:", HF_CACHE)


Mounted at /content/drive
HF cache: /content/drive/MyDrive/hf_cache


In [3]:
import os
import math
import torch
import torchaudio
from tqdm import tqdm
from datasets import load_dataset, interleave_datasets, DatasetDict


In [4]:
EASTERN_EUROPEAN_CONFIGS = [
    "hy_am","be_by","bg_bg","cs_cz","et_ee","ka_ge","lv_lv","lt_lt",
    "mk_mk","pl_pl","ro_ro","ru_ru","sr_rs","sk_sk","sl_si","uk_ua"
]

# Global lang_id values for those configs (from your group file)
EASTERN_EUROPEAN_GLOBAL_IDS = [35, 6, 7, 14, 21, 42, 56, 54, 58, 74, 77, 78, 84, 80, 81, 92]

global_to_local = {gid: i for i, gid in enumerate(EASTERN_EUROPEAN_GLOBAL_IDS)}
num_classes = len(EASTERN_EUROPEAN_CONFIGS)

print("Num EE languages:", num_classes)
print("Example global->local:", list(global_to_local.items())[:5])


Num EE languages: 16
Example global->local: [(35, 0), (6, 1), (7, 2), (14, 3), (21, 4)]


In [5]:
def load_group_dataset_non_streaming(configs):
    ds_list = []
    for cfg in configs:
        print(f"Loading config: {cfg}")
        ds_list.append(load_dataset("google/fleurs", cfg))  # non-streaming; cached to Drive

    combined = DatasetDict({
        split: interleave_datasets([ds[split] for ds in ds_list])
        for split in ds_list[0].keys()
    })

    # shuffle train/val for better mixing
    combined["train"] = combined["train"].shuffle(seed=42)
    combined["validation"] = combined["validation"].shuffle(seed=42)

    return combined

ee_ds = load_group_dataset_non_streaming(EASTERN_EUROPEAN_CONFIGS)
print(ee_ds)


Loading config: hy_am


README.md: 0.00B [00:00, ?B/s]

fleurs.py: 0.00B [00:00, ?B/s]

The repository for google/fleurs contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/google/fleurs.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N] y


data/hy_am/audio/train.tar.gz:   0%|          | 0.00/1.73G [00:00<?, ?B/s]

...

test.tsv: 0.00B [00:00, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'num_samples', 'path', 'audio', 'transcription', 'raw_transcription', 'gender', 'lang_id', 'language', 'lang_group_id'],
        num_rows: 23856
    })
    validation: Dataset({
        features: ['id', 'num_samples', 'path', 'audio', 'transcription', 'raw_transcription', 'gender', 'lang_id', 'language', 'lang_group_id'],
        num_rows: 4640
    })
    test: Dataset({
        features: ['id', 'num_samples', 'path', 'audio', 'transcription', 'raw_transcription', 'gender', 'lang_id', 'language', 'lang_group_id'],
        num_rows: 10528
    })
})


In [6]:
TARGET_SR = 16000
N_MELS = 80

mel = torchaudio.transforms.MelSpectrogram(
    sample_rate=TARGET_SR,
    n_fft=400,
    hop_length=160,
    n_mels=N_MELS,
)
amp_to_db = torchaudio.transforms.AmplitudeToDB()

def waveform_to_logmel(waveform_1xN: torch.Tensor) -> torch.Tensor:
    m = mel(waveform_1xN)                         # [1, 80, T]
    lm = amp_to_db(m)                             # [1, 80, T]
    return lm.squeeze(0).transpose(0, 1).contiguous()  # [T, 80]


In [7]:
SAVE_ROOT = "/content/drive/MyDrive/fleurs_preprocessed/eastern_european"
os.makedirs(SAVE_ROOT, exist_ok=True)

def preprocess_and_save_shards(split_ds, shard_dir, prefix, shard_size=500):
    os.makedirs(shard_dir, exist_ok=True)

    shard = []
    shard_idx = 0
    total = 0

    for ex in tqdm(split_ds):
        wav = torch.tensor(ex["audio"]["array"], dtype=torch.float32)
        gid = int(ex["lang_id"])
        lid = global_to_local[gid]  # 0..15

        x = waveform_to_logmel(wav.unsqueeze(0))  # [T,80]

        shard.append({
            "x": x,
            "y": lid,
            "global_lang_id": gid,
            "language": ex.get("language", "N/A"),
        })
        total += 1

        if len(shard) >= shard_size:
            out_path = os.path.join(shard_dir, f"{prefix}_shard_{shard_idx:03d}.pt")
            torch.save(shard, out_path)
            shard = []
            shard_idx += 1

    # save remainder
    if shard:
        out_path = os.path.join(shard_dir, f"{prefix}_shard_{shard_idx:03d}.pt")
        torch.save(shard, out_path)

    print(f"Saved {total} examples into shards at: {shard_dir}")

train_shards = os.path.join(SAVE_ROOT, "train_shards")
val_shards   = os.path.join(SAVE_ROOT, "validation_shards")
test_shards  = os.path.join(SAVE_ROOT, "test_shards")

preprocess_and_save_shards(ee_ds["train"], train_shards, "train", shard_size=500)
preprocess_and_save_shards(ee_ds["validation"], val_shards, "validation", shard_size=500)
preprocess_and_save_shards(ee_ds["test"], test_shards, "test", shard_size=500)

100%|██████████| 23856/23856 [08:35<00:00, 46.31it/s]


Saved 23856 examples into shards at: /content/drive/MyDrive/fleurs_preprocessed/eastern_european/train_shards


100%|██████████| 4640/4640 [01:37<00:00, 47.76it/s]


Saved 4640 examples into shards at: /content/drive/MyDrive/fleurs_preprocessed/eastern_european/validation_shards


100%|██████████| 10528/10528 [03:03<00:00, 57.25it/s]


Saved 10528 examples into shards at: /content/drive/MyDrive/fleurs_preprocessed/eastern_european/test_shards
